# Hyperliquid × Bitcoin Fear/Greed Index
### Trader Behavior & Performance Analysis - How Market Sentiment Drives Trader Behavior on Hyperliquid

## Data Cleaning

**Goal:** Prepare the working dataset for analysis by removing structurally
invalid records, filtering noise trade types, and standardising categories.

**Why DeFi cleaning goes beyond standard tabular cleaning:**  
Standard cleaning removes nulls and duplicates. DeFi trade cleaning additionally
requires: separating perpetual futures from spot trades (different mechanics),
filtering system-generated events (dust conversions, auto-deleveraging),
and confirming that PnL is only attributed to closing events.

**Two-part approach:**
```
Part 1 — Structural Cleaning
Drop null sentiment rows → Remove non-analytical trade types
→ Validate dtypes → Confirm no row-level duplicates

Part 2 — Semantic Cleaning
Standardise Account labels → Validate PnL attribution
→ Cap leverage outliers → Export clean_trades.csv
```

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('working_trades.csv')
print('Input shape:', df.shape)
df.head(3)

Input shape: (211224, 21)


,Account,Coin,Execution Price,Size Tokens,Size USD,Side,date,Start Position,Direction,Closed PnL,...,Fee,fg_value,sentiment,sentiment_binary,net_pnl,is_close,is_win,direction_clean,leverage_proxy,size_bucket
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,2024-12-02,0.000000,Buy,0.0,...,0.345404,80.0,Extreme Greed,Greed,-0.345404,False,False,spot_buy,NaN,large
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,2024-12-02,986.524596,Buy,0.0,...,0.005600,80.0,Extreme Greed,Greed,-0.005600,False,False,spot_buy,0.129424,small
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,2024-12-02,1002.518996,Buy,0.0,...,0.050431,80.0,Extreme Greed,Greed,-0.050431,False,False,spot_buy,1.147739,medium


### Pre-Cleaning Snapshot

In [2]:
df.isnull().sum()

Account                0
Coin                   0
Execution Price        0
Size Tokens            0
Size USD               0
Side                   0
date                   0
Start Position         0
Direction              0
Closed PnL             0
Crossed                0
Fee                    0
fg_value               6
sentiment              6
sentiment_binary       6
net_pnl                0
is_close               0
is_win                 0
direction_clean        2
leverage_proxy      4085
size_bucket           43
dtype: int64

In [3]:
print('Row duplicates:', df.duplicated().sum())
print('Direction types present:', df['direction_clean'].value_counts().to_dict())

Row duplicates: 0
Direction types present: {'open_long': 49895, 'close_long': 48678, 'open_short': 39741, 'close_short': 36013, 'spot_sell': 19902, 'spot_buy': 16716, 'dust': 142, 'flip': 127, 'adl': 8}


**Pre-cleaning null snapshot:**

Only two columns carry nulls — `sentiment` / `sentiment_binary` (6 rows with unmatched date)
and `leverage_proxy` (undefined when Start Position = 0, which is expected for new positions).
The leverage proxy nulls are structural, not errors — we retain those rows.

Our cleaning decisions:
- `sentiment` nulls → Remove. No sentiment label = cannot classify the trade.
- `leverage_proxy` nulls → Retain. These are first-entry trades, valid for all other metrics.
- `size_bucket` nulls (Size USD = 0) → Investigate below.

### Step 1 — Remove Null Sentiment Rows

In [4]:
before = len(df)
df = df.dropna(subset=['sentiment'])
print(f'Removed {before - len(df)} rows with no sentiment label')
print(f'Remaining: {len(df)}')

Removed 6 rows with no sentiment label
Remaining: 211218


### Step 2 — Remove Non-Analytical Trade Types

Two direction types add noise without analytical value:
- `dust` (Spot Dust Conversion): System-generated micro-trades, not trader decisions.
- `adl` (Auto-Deleveraging): Force-closes triggered by the exchange, not the trader.

We flag and remove these, keeping spot and flip trades which represent intentional decisions.

In [5]:
noise_types = ['dust', 'adl']
before = len(df)
df = df[~df['direction_clean'].isin(noise_types)]
print(f'Removed {before - len(df)} noise-type trades (dust + ADL)')
print(f'Remaining: {len(df)}')
print()
print('Direction types after filter:')
df['direction_clean'].value_counts()

Removed 150 noise-type trades (dust + ADL)
Remaining: 211068

Direction types after filter:


direction_clean
open_long      49895
close_long     48678
open_short     39741
close_short    36007
spot_sell      19902
spot_buy       16716
flip             127
Name: count, dtype: int64

### Step 3 — Investigate Zero-Size Trades

In [6]:
zero_size = df[df['Size USD'] == 0]
print('Trades with Size USD = 0:', len(zero_size))
print()
if len(zero_size) > 0:
    print('Direction types of zero-size trades:')
    print(zero_size['direction_clean'].value_counts())
    print()
    print('PnL on zero-size trades:')
    print(zero_size['Closed PnL'].describe())

Trades with Size USD = 0: 0



In [7]:
# Decision: zero-size trades with zero PnL are operational artifacts — remove
# Zero-size trades with non-zero PnL are settlement events — keep
remove_zero = (df['Size USD'] == 0) & (df['Closed PnL'] == 0)
before = len(df)
df = df[~remove_zero]
print(f'Removed {before - len(df)} zero-size zero-PnL trades')
print(f'Remaining: {len(df)}')

Removed 0 zero-size zero-PnL trades
Remaining: 211068


### Step 4 — Validate Data Types

In [8]:
# Convert date back to datetime for time-series operations
df['date'] = pd.to_datetime(df['date'])

# Ensure numeric columns are float
numeric_cols = ['Execution Price', 'Size Tokens', 'Size USD', 'Closed PnL', 'Fee', 'net_pnl']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Confirm
df[numeric_cols].dtypes

Execution Price    float64
Size Tokens        float64
Size USD           float64
Closed PnL         float64
Fee                float64
net_pnl            float64
dtype: object

### Step 5 — Standardise Account Labels

Account addresses are long hex strings — hard to read in charts.
We create short labels (Trader_01 ... Trader_16) ordered by trade count descending.

In [9]:
account_order = df['Account'].value_counts().index.tolist()
account_labels = {acc: f'Trader_{str(i+1).zfill(2)}' for i, acc in enumerate(account_order)}
df['trader_id'] = df['Account'].map(account_labels)

print('Account → Trader ID mapping:')
for k, v in account_labels.items():
    print(f'  {v}: {k[:12]}...')

Account → Trader ID mapping:
  Trader_01: 0xbee1707d6b...
  Trader_02: 0xbaaaf6571a...
  Trader_03: 0xa0feb3725a...
  Trader_04: 0x8477e44784...
  Trader_05: 0xb1231a4a2d...
  Trader_06: 0x28736f43f1...
  Trader_07: 0x513b8629fe...
  Trader_08: 0x75f7eeb85d...
  Trader_09: 0x47add9a56d...
  Trader_10: 0x4f93fead39...
  Trader_11: 0x23e7a7f8d1...
  Trader_12: 0xb899e522b5...
  Trader_13: 0x8170715b3b...
  Trader_14: 0x4acb90e786...
  Trader_15: 0x083384f897...
  Trader_16: 0x271b280974...
  Trader_17: 0x39cef799f8...
  Trader_18: 0x2c229d22b1...
  Trader_19: 0x92f17e8d81...
  Trader_20: 0xbd5fead718...
  Trader_21: 0x8381e6d82f...
  Trader_22: 0x72743ae282...
  Trader_23: 0x7f4f299f74...
  Trader_24: 0x72c6a4624e...
  Trader_25: 0x430f09841d...
  Trader_26: 0x6d6a4b953f...
  Trader_27: 0x3998f134d6...
  Trader_28: 0xae5eacaf9c...
  Trader_29: 0xaf40fdc468...
  Trader_30: 0xa520ded057...
  Trader_31: 0x420ab45e0b...
  Trader_32: 0x3f9a0aadc7...


### Step 6 — Cap Leverage Proxy Outliers

The leverage proxy can produce extreme values when Start Position is very small.
We already capped at 100x during extraction. Here we verify the cap held.

In [10]:
lev = df['leverage_proxy'].dropna()
print('Leverage proxy describe:')
print(lev.describe())
print()
print('Values at or above 100x cap:', (lev >= 100).sum())

Leverage proxy describe:
count    2.069830e+05
mean     1.580125e+01
std      3.408301e+01
min      9.570242e-09
25%      1.659490e-02
50%      1.804883e-01
75%      3.104833e+00
max      1.000000e+02
Name: leverage_proxy, dtype: float64

Values at or above 100x cap: 26665


### Step 7 — Validate PnL Attribution

PnL should only be non-zero on closing trades. We check for anomalies.

In [11]:
# Non-close trades with non-zero PnL
pnl_on_opens = df[~df['is_close'] & (df['Closed PnL'] != 0)]
print('Non-close trades with non-zero PnL:', len(pnl_on_opens))
if len(pnl_on_opens) > 0:
    print('Direction types:')
    print(pnl_on_opens['direction_clean'].value_counts())
    print('Total PnL on these rows:', pnl_on_opens['Closed PnL'].sum().round(2))

Non-close trades with non-zero PnL: 19774
Direction types:
direction_clean
spot_sell    19646
flip           126
Name: count, dtype: int64
Total PnL on these rows: 2906751.0


Non-zero PnL on non-close trades is expected for `flip` direction types
(Long > Short reverses the position and realizes partial PnL simultaneously).
These rows are valid and retained.

### Post-Cleaning Summary

In [12]:
print('=== Post-Cleaning Dataset ===' )
print(f'Shape: {df.shape}')
print()
print('Null counts:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print()
print('Sentiment distribution:')
print(df['sentiment'].value_counts())
print()
print('Trader distribution:')
print(df['trader_id'].value_counts())

=== Post-Cleaning Dataset ===
Shape: (211068, 22)

Null counts:
direction_clean       2
leverage_proxy     4085
dtype: int64

Sentiment distribution:
sentiment
Fear             61795
Greed            50240
Extreme Greed    39960
Neutral          37676
Extreme Fear     21397
Name: count, dtype: int64

Trader distribution:
trader_id
Trader_01    40172
Trader_02    21192
Trader_03    15604
Trader_04    14994
Trader_05    14722
Trader_06    13301
Trader_07    12236
Trader_08     9893
Trader_09     8513
Trader_10     7562
Trader_11     7271
Trader_12     4838
Trader_13     4601
Trader_14     4355
Trader_15     3818
Trader_16     3807
Trader_17     3589
Trader_18     3239
Trader_19     3051
Trader_20     2641
Trader_21     1911
Trader_22     1590
Trader_23     1559
Trader_24     1424
Trader_25     1215
Trader_26      972
Trader_27      780
Trader_28      562
Trader_29      534
Trader_30      411
Trader_31      383
Trader_32      328
Name: count, dtype: int64


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 211068 entries, 0 to 211223
Data columns (total 22 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Account           211068 non-null  object        
 1   Coin              211068 non-null  object        
 2   Execution Price   211068 non-null  float64       
 3   Size Tokens       211068 non-null  float64       
 4   Size USD          211068 non-null  float64       
 5   Side              211068 non-null  object        
 6   date              211068 non-null  datetime64[ns]
 7   Start Position    211068 non-null  float64       
 8   Direction         211068 non-null  object        
 9   Closed PnL        211068 non-null  float64       
 10  Crossed           211068 non-null  bool          
 11  Fee               211068 non-null  float64       
 12  fg_value          211068 non-null  float64       
 13  sentiment         211068 non-null  object        
 14  sentiment

### Export Clean Dataset

In [14]:
df.to_csv('clean_trades.csv', index=False)
print('Exported clean_trades.csv')
print('Final shape:', df.shape)

Exported clean_trades.csv
Final shape: (211068, 22)


## Cleaning Summary

| Step | Action | Rows Removed | Remaining |
|---|---|---|---|
| Start | Working dataset | — | 2,11,224 |
| 1 | Remove null sentiment | 6 | 2,11,218 |
| 2 | Remove dust + ADL trades | ~150 | 2,11,068 |
| 3 | Remove zero-size, zero-PnL | - | 2,11,068 |

**Retained:** ~99.8% of original records

**Added columns:** `trader_id` (readable account labels)

**Output file:** `clean_trades.csv`

**Next:** Notebook 04 — EDA.